# pcode2cpp | Spore retail (WSL)

Run the **first code cell** after opening or restarting the notebook. It initializes configuration, opens the existing Ghidra project, and exports the requested retail function to `cases/<address>/context.json`. If the function is missing because the binary has not been analyzed, the exporter performs full Ghidra auto-analysis **once**, saves the same project, and exports. Subsequent exports reuse the saved analysis.

Everything runs in WSL. The retail binary must already be imported into the configured Ghidra project; no new project or binary import occurs. The full first analysis can take substantially longer than later exports. Model inference and verification remain separate cells.

In [ ]:
from pathlib import Path
import json
import sys

# This cell is self-contained. Run it after a Jupyter kernel restart.
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Launch JupyterLab from the pcode2cpp repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pcode2cpp.ghidra import export_function
from pcode2cpp.translate import build_messages, translate
from integrations.spore_decomp import SporeDecompBackend

CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
ADDRESS = 0x00672CF0  # Change this to another retail function entry address.
CASE = ROOT / "cases" / f"{ADDRESS:08x}"
CASE.mkdir(parents=True, exist_ok=True)

print(f"Project: {CONFIG['ghidra']['project_name']} / {CONFIG['ghidra'].get('project_program', 'configured program')}", flush=True)
print(f"Retail address: 0x{ADDRESS:08X}", flush=True)
print(f"Context output: {CASE / 'context.json'}", flush=True)

# Export from the existing project. Missing functions trigger full analysis
# and persistence of the analyzed project, without manual notebook edits.
context = export_function(ADDRESS, CASE / "context.json", CONFIG["ghidra"])
print("Function:", context["function"]["name"])
print("High P-code operations:", len(context["high_pcode"]))
print("Assembly instructions:", len(context["assembly"]))
print("Calls:", len(context["calls"]), "Referenced types:", len(context["types"]))

## 1. Inspect the extracted retail function

Review its identity, calls, types and decompiled C. The complete context remains in `cases/<address>/context.json`; previews here do not alter the inference data.

In [ ]:
print(json.dumps(context["function"], indent=2))
print("\nCALLS:\n", json.dumps(context["calls"], indent=2))
print("\nTYPES:\n", json.dumps(context["types"], indent=2)[:12000])
print("\nDECOMPILED C:\n", context["pseudocode"])
print("\nHIGH PCODE (first 20 for display only):\n", json.dumps(context["high_pcode"][:20], indent=2))


## 2. Inspect the complete prompt

The model receives **all** exported fields, without truncation. Check that the llama.cpp context window can hold the request and output.

In [ ]:
messages = build_messages(context)
print("SYSTEM INSTRUCTION:\n", messages[0]["content"])
print("\nFULL USER CONTEXT:\n", messages[1]["content"])
print("\nInput characters:", sum(len(m["content"]) for m in messages))


## 3. Generate C++ with local Inkling

This cell makes one tool-free local model request and saves `candidate.cpp` and `inference.json`.

In [ ]:
cpp, metadata = translate(context, CONFIG["llm"])
(CASE / "candidate.cpp").write_text(cpp, encoding="utf-8")
(CASE / "inference.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(cpp)
print("\nInference metadata:", metadata)


## 4. Optional Spore verification

Verification uses a manually prepared translation unit in a separate Spore checkout. This cell does not edit Spore source files.

In [ ]:
SPORE_SOURCE = None  # e.g. Path(CONFIG["spore"]["repo_path"]) / "match/slices/.../...cpp"
if SPORE_SOURCE is None:
    print("Skipped: set SPORE_SOURCE to the prepared translation unit to verify.")
else:
    spore = CONFIG["spore"]
    backend = SporeDecompBackend(Path(spore["repo_path"]), spore.get("python", ".venv/bin/python"), spore.get("timeout_seconds", 600))
    result = backend.verify(ADDRESS, Path(SPORE_SOURCE))
    (CASE / "verification.txt").write_text(result.verdict.value + "\n" + result.details + "\n" + result.compiler_output, encoding="utf-8")
    print("Verdict:", result.verdict.value, "Proof layer:", result.proof_method)
    print(result.details)
